In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

bronze_df = spark.table(
    "customer360_dev.bronze.order_items"
)

orders_df = (
    spark.table("customer360_dev.silver.orders")
    .select("order_id")
)

products_df = (
    spark.table("customer360_dev.silver.products")
    .select("product_id")
)

In [0]:
typed_df = (
    bronze_df
    .withColumn(
        "quantity_cast",
        F.col("quantity").cast("int")
    )
    .withColumn(
        "unit_price_cast",
        F.col("unit_price").cast("decimal(14,2)")
    )
    .withColumn(
        "discount_amount_cast",
        F.col("discount_amount").cast("decimal(14,2)")
    )
    .withColumn(
        "tax_amount_cast",
        F.col("tax_amount").cast("decimal(14,2)")
    )
    .withColumn(
        "line_amount_cast",
        F.col("line_amount").cast("decimal(14,2)")
    )
    .withColumn(
        "item_status_clean",
        F.upper(F.trim("item_status"))
    )
    .withColumn(
        "created_at_cast",
        F.to_timestamp("created_at")
    )
    .withColumn(
        "updated_at_cast",
        F.to_timestamp("updated_at")
    )
)

In [0]:
validated_df = (
    typed_df
    .join(
        orders_df.withColumn(
            "_order_exists",
            F.lit(True)
        ),
        on="order_id",
        how="left"
    )
    .join(
        products_df.withColumn(
            "_product_exists",
            F.lit(True)
        ),
        on="product_id",
        how="left"
    )
)

Define core DQ rules

In [0]:
validated_df = (
    validated_df
    .withColumn(
        "dq_missing_order_item_id",
        F.col("order_item_id").isNull()
        | (F.trim(F.col("order_item_id")) == "")
    )
    .withColumn(
        "dq_missing_order_id",
        F.col("order_id").isNull()
        | (F.trim(F.col("order_id")) == "")
    )
    .withColumn(
        "dq_missing_product_id",
        F.col("product_id").isNull()
        | (F.trim(F.col("product_id")) == "")
    )
    .withColumn(
        "dq_order_not_found",
        F.col("_order_exists").isNull()
    )
    .withColumn(
        "dq_product_not_found",
        F.col("_product_exists").isNull()
    )
    .withColumn(
        "dq_invalid_quantity",
        F.col("quantity_cast").isNull()
        | (F.col("quantity_cast") <= 0)
    )
    .withColumn(
        "dq_invalid_unit_price",
        F.col("unit_price_cast").isNull()
        | (F.col("unit_price_cast") < 0)
    )
    .withColumn(
        "dq_invalid_discount_amount",
        F.col("discount_amount_cast").isNull()
        | (F.col("discount_amount_cast") < 0)
    )
    .withColumn(
        "dq_invalid_tax_amount",
        F.col("tax_amount_cast").isNull()
        | (F.col("tax_amount_cast") < 0)
    )
    .withColumn(
        "dq_invalid_line_amount",
        F.col("line_amount_cast").isNull()
        | (F.col("line_amount_cast") < 0)
    )
    .withColumn(
        "dq_invalid_item_status",
        ~F.col("item_status_clean").isin(
            "CREATED",
            "CONFIRMED",
            "SHIPPED",
            "DELIVERED",
            "CANCELLED",
            "RETURNED"
        )
    )
)

Add line-level financial reconciliation

In [0]:
validated_df = (
    validated_df
    .withColumn(
        "calculated_line_amount",
        (
            (
                F.col("quantity_cast")
                * F.col("unit_price_cast")
            )
            - F.col("discount_amount_cast")
            + F.col("tax_amount_cast")
        ).cast("decimal(14,2)")
    )
    .withColumn(
        "dq_line_amount_mismatch",
        F.abs(
            F.col("calculated_line_amount")
            - F.col("line_amount_cast")
        ) > F.lit(0.01)
    )
)

Add warning rules

In [0]:
validated_df = (
    validated_df
    .withColumn(
        "dq_warning_discount_exceeds_line_gross",
        F.col("discount_amount_cast")
        > (
            F.col("quantity_cast")
            * F.col("unit_price_cast")
        )
    )
)

Define critical failure

In [0]:
validated_df = validated_df.withColumn(
    "dq_critical_failure",
    F.col("dq_missing_order_item_id")
    | F.col("dq_missing_order_id")
    | F.col("dq_missing_product_id")
    | F.col("dq_order_not_found")
    | F.col("dq_product_not_found")
    | F.col("dq_invalid_quantity")
    | F.col("dq_invalid_unit_price")
    | F.col("dq_invalid_discount_amount")
    | F.col("dq_invalid_tax_amount")
    | F.col("dq_invalid_line_amount")
    | F.col("dq_invalid_item_status")
    | F.col("dq_line_amount_mismatch")
)

Split valid and rejected records

In [0]:
reject_df = validated_df.filter(
    F.col("dq_critical_failure")
)

valid_df = validated_df.filter(
    ~F.col("dq_critical_failure")
)

Deduplicate deterministically

In [0]:
order_item_window = (
    Window
    .partitionBy("order_item_id")
    .orderBy(
        F.col("updated_at_cast").desc_nulls_last(),
        F.col("_ingestion_timestamp").desc()
    )
)

deduped_df = (
    valid_df
    .withColumn(
        "_row_number",
        F.row_number().over(order_item_window)
    )
    .filter(F.col("_row_number") == 1)
)

Build final Silver Order Items structure

In [0]:
silver_df = deduped_df.select(
    "order_item_id",
    "order_id",
    "product_id",

    F.col("quantity_cast").alias("quantity"),
    F.col("unit_price_cast").alias("unit_price"),
    F.col("discount_amount_cast").alias("discount_amount"),
    F.col("tax_amount_cast").alias("tax_amount"),
    F.col("line_amount_cast").alias("line_amount"),

    F.col("item_status_clean").alias("item_status"),

    "dq_warning_discount_exceeds_line_gross",

    F.col("created_at_cast").alias("created_at"),
    F.col("updated_at_cast").alias("updated_at"),

    "_source_file",
    "_source_system",
    "_load_type",
    "_ingestion_timestamp",

    F.current_timestamp().alias(
        "_silver_processed_at"
    )
)

In [0]:
(
    silver_df
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(
        "customer360_dev.silver.order_items"
    )
)

Build quarantine records

In [0]:
quarantine_df = (
    reject_df
    .withColumn(
        "rejection_reason",
        F.concat_ws(
            "; ",
            F.when(
                F.col("dq_missing_order_item_id"),
                F.lit("MISSING_ORDER_ITEM_ID")
            ),
            F.when(
                F.col("dq_missing_order_id"),
                F.lit("MISSING_ORDER_ID")
            ),
            F.when(
                F.col("dq_missing_product_id"),
                F.lit("MISSING_PRODUCT_ID")
            ),
            F.when(
                F.col("dq_order_not_found"),
                F.lit("ORDER_NOT_FOUND")
            ),
            F.when(
                F.col("dq_product_not_found"),
                F.lit("PRODUCT_NOT_FOUND")
            ),
            F.when(
                F.col("dq_invalid_quantity"),
                F.lit("INVALID_QUANTITY")
            ),
            F.when(
                F.col("dq_invalid_unit_price"),
                F.lit("INVALID_UNIT_PRICE")
            ),
            F.when(
                F.col("dq_invalid_discount_amount"),
                F.lit("INVALID_DISCOUNT_AMOUNT")
            ),
            F.when(
                F.col("dq_invalid_tax_amount"),
                F.lit("INVALID_TAX_AMOUNT")
            ),
            F.when(
                F.col("dq_invalid_line_amount"),
                F.lit("INVALID_LINE_AMOUNT")
            ),
            F.when(
                F.col("dq_invalid_item_status"),
                F.lit("INVALID_ITEM_STATUS")
            ),
            F.when(
                F.col("dq_line_amount_mismatch"),
                F.lit("LINE_AMOUNT_MISMATCH")
            )
        )
    )
    .withColumn(
        "_quarantined_at",
        F.current_timestamp()
    )
)

In [0]:
if "_pipeline_run_id" not in spark.table("customer360_dev.quarantine.order_item_rejects").columns:
    spark.sql("ALTER TABLE customer360_dev.quarantine.order_item_rejects ADD COLUMNS (_pipeline_run_id STRING)")
(
    quarantine_df
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(
        "customer360_dev.quarantine.order_item_rejects"
    )
)

In [0]:
bronze_count = spark.table(
    "customer360_dev.bronze.order_items"
).count()

silver_count = spark.table(
    "customer360_dev.silver.order_items"
).count()

reject_count = spark.table(
    "customer360_dev.quarantine.order_item_rejects"
).count()

print(f"Bronze Order Items : {bronze_count:,}")
print(f"Silver Order Items : {silver_count:,}")
print(f"Rejected           : {reject_count:,}")

Bronze Order Items : 15,000,186
Silver Order Items : 11,400,000
Rejected           : 3,600,186
